# Team Members

1. Kiran Kumar GANESAN
2. Rithiga VENGADESSANE


In [1]:
import sys
! "{sys.executable}" -m pip install rank-bm25 nltk scikit-learn pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Ganesan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load Data
places = pd.read_csv('Dataset/Tripadvisor.csv')
reviews = pd.read_csv('Dataset/reviews83325.csv')

# 2. Filter for English reviews only
reviews = reviews[reviews['langue'] == 'en'].copy()

# 3. Create a "Place Profile"
place_profiles = reviews.groupby('idplace')['review'].apply(lambda x: ' '.join(str(v) for v in x)).reset_index()

# 4. Merge text profiles with metadata
df = pd.merge(place_profiles, places, left_on='idplace', right_on='id')

# 5. Split 50/50 for Train/Test
train_df, test_df = train_test_split(df, test_size=0.5, random_state=42)

print(f"Training on {len(train_df)} places. Testing on {len(test_df)} places.")

C:\Users\Ganesan\AppData\Local\Temp\ipykernel_2016\367400263.py:7: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv('Dataset/reviews83325.csv')


Training on 917 places. Testing on 918 places.


In [3]:
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize

# Download necessary tokenizer data
nltk.download('punkt')
nltk.download('punkt_tab')

# 1. Tokenize corpus
print("Tokenizing training data for BM25...")
tokenized_corpus = [word_tokenize(doc.lower()) for doc in train_df['review'].tolist()]

# 2. Initialize BM25
bm25 = BM25Okapi(tokenized_corpus)

def get_bm25_ranks(query_text):
    tokenized_query = word_tokenize(query_text.lower())
    scores = bm25.get_scores(tokenized_query)
    return np.argsort(scores)[::-1]


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Ganesan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Ganesan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Tokenizing training data for BM25...


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Step 4: TF-IDF + cosine
print("Building TF-IDF matrix on training data...")
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix_train = tfidf.fit_transform(train_df['review'])


def get_tfidf_ranks(query_text):
    query_vec = tfidf.transform([query_text])
    similarities = cosine_similarity(query_vec, tfidf_matrix_train).flatten()
    return np.argsort(similarities)[::-1]

# Step 5: ranking error metric
def calculate_ranking_error(recommended_indices, query_row, train_df, level=1):
    for rank, idx in enumerate(recommended_indices):
        match_found = False
        target_place = train_df.iloc[idx]
        if level == 1:
            if target_place['typeR'] == query_row['typeR']:
                match_found = True
        else:
            query_cats = str(query_row.get('activiteSubCategorie', '')).split(',')
            target_cats = str(target_place.get('activiteSubCategorie', '')).split(',')
            if any(cat in target_cats for cat in query_cats if cat and cat != 'nan'):
                match_found = True
        if match_found:
            return rank
    return None

print("TF-IDF and ranking helper functions defined.")


Building TF-IDF matrix on training data...
TF-IDF and ranking helper functions defined.


In [5]:
# Evaluation loop

bm25_errors = []
tfidf_errors = []
sample_test = test_df.sample(100, random_state=42)

for _, query_row in sample_test.iterrows():
    bm25_ranks = get_bm25_ranks(query_row['review'])
    tfidf_ranks = get_tfidf_ranks(query_row['review'])
    err_bm25 = calculate_ranking_error(bm25_ranks, query_row, train_df, level=1)
    err_tfidf = calculate_ranking_error(tfidf_ranks, query_row, train_df, level=1)
    if err_bm25 is not None:
        bm25_errors.append(err_bm25)
    if err_tfidf is not None:
        tfidf_errors.append(err_tfidf)

print(f"Average Ranking Error (BM25): {np.mean(bm25_errors):.2f}")
print(f"Average Ranking Error (TF-IDF): {np.mean(tfidf_errors):.2f}")


Average Ranking Error (BM25): 1.08
Average Ranking Error (TF-IDF): 0.86


In [6]:
# Level 2 evaluation

bm25_errors_lvl2 = []
tfidf_errors_lvl2 = []

for _, query_row in sample_test.iterrows():
    bm25_ranks = get_bm25_ranks(query_row['review'])
    tfidf_ranks = get_tfidf_ranks(query_row['review'])

    err_bm25 = calculate_ranking_error(bm25_ranks, query_row, train_df, level=2)
    err_tfidf = calculate_ranking_error(tfidf_ranks, query_row, train_df, level=2)

    if err_bm25 is not None:
        bm25_errors_lvl2.append(err_bm25)
    if err_tfidf is not None:
        tfidf_errors_lvl2.append(err_tfidf)

print("\nLevel‑2 (subcategory/cuisine) average errors:")
print(f"BM25: {np.mean(bm25_errors_lvl2):.2f}, TF-IDF: {np.mean(tfidf_errors_lvl2):.2f}")



Level‑2 (subcategory/cuisine) average errors:
BM25: 56.62, TF-IDF: 17.75


In [7]:
# Display sample recommendations

sample_output = []
for i, query_row in sample_test.head(5).iterrows():
    qtext = query_row['review']
    bm25_idx = get_bm25_ranks(qtext)[:5]
    tfidf_idx = get_tfidf_ranks(qtext)[:5]
    sample_output.append({
        'query_id': query_row['idplace'],
        'query_type': query_row['typeR'],
        'bm25_top5': list(train_df.iloc[bm25_idx][['idplace','typeR']].to_records(index=False)),
        'tfidf_top5': list(train_df.iloc[tfidf_idx][['idplace','typeR']].to_records(index=False))
    })

import pandas as pd
print("Average Ranking Errors")
print(f"BM25: {np.mean(bm25_errors):.2f}, TF-IDF: {np.mean(tfidf_errors):.2f}")

display(pd.DataFrame(sample_output))


Average Ranking Errors
BM25: 1.08, TF-IDF: 0.86


,query_id,query_type,bm25_top5,tfidf_top5
0,3573554,R,"[[799588, R], [695212, R], [714984, R], [69518...","[[14068772, R], [4063323, R], [2035323, R], [1..."
1,19150888,AP,"[[11860584, AP], [4064976, A], [11860583, AP],...","[[11860584, AP], [11860586, AP], [4064976, A],..."
2,1024865,R,"[[1863209, R], [209761, A], [5820233, R], [389...","[[5820233, R], [1863209, R], [8394653, R], [41..."
3,2051752,R,"[[718188, R], [858456, R], [1524417, R], [6952...","[[858456, R], [718188, R], [1335980, R], [1524..."
4,19878182,AP,"[[2479112, A], [2076448, A], [16725139, AP], [...","[[12923667, AP], [17814115, AP], [12923648, AP..."


In [8]:
# --- STEP 7: FINAL DASHBOARD SUMMARY ---
import pandas as pd

summary_data = {
    "Metric": ["Level 1 Error (General Type)", "Level 2 Error (Specific Details)"],
    "BM25 (Baseline)": [f"{np.mean(bm25_errors):.2f}", f"{np.mean(bm25_errors_lvl2):.2f}"],
    "TF-IDF (Adv Model)": [f"{np.mean(tfidf_errors):.2f}", f"{np.mean(tfidf_errors_lvl2):.2f}"]
}

print("-" * 35)
print("  PROJECT PERFORMANCE DASHBOARD")
print("-" * 35)
display(pd.DataFrame(summary_data))

# Calculate the improvement percentage for the report
imp = ((np.mean(bm25_errors_lvl2) - np.mean(tfidf_errors_lvl2)) / np.mean(bm25_errors_lvl2)) * 100
print(f"\nFinal Result: The TF-IDF model reduced specific error by {imp:.1f}%.")

-----------------------------------
  PROJECT PERFORMANCE DASHBOARD
-----------------------------------


,Metric,BM25 (Baseline),TF-IDF (Adv Model)
0,Level 1 Error (General Type),1.08,0.86
1,Level 2 Error (Specific Details),56.62,17.75



Final Result: The TF-IDF model reduced specific error by 68.7%.
